# Clase 204 — Shadow + Canary + A/B test

Simulamos los 3 patrones en proceso (sin Istio), con sticky assignment, métricas, rollback automático y análisis estadístico del A/B.

## Setup — 2 modelos: champion vs challenger

In [ ]:
import numpy as np, hashlib, time, random
from collections import defaultdict
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)

champion = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
challenger = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr, ytr)
print(f'champion offline acc: {accuracy_score(yte, champion.predict(Xte)):.4f}')
print(f'challenger offline acc: {accuracy_score(yte, challenger.predict(Xte)):.4f}')

## 1. Shadow mode (challenger no responde)

In [ ]:
shadow_log = []

def predict_with_shadow(row, true_label):
    t0 = time.perf_counter()
    champ_pred = int(champion.predict(row.reshape(1, -1))[0])
    t_champ = time.perf_counter() - t0
    t1 = time.perf_counter()
    chal_pred = int(challenger.predict(row.reshape(1, -1))[0])
    t_chal = time.perf_counter() - t1
    shadow_log.append({
        'champ_pred': champ_pred, 'chal_pred': chal_pred, 'truth': int(true_label),
        'champ_lat_ms': t_champ * 1000, 'chal_lat_ms': t_chal * 1000,
    })
    return champ_pred   # solo champion responde al usuario

for i in range(len(Xte)):
    predict_with_shadow(Xte[i], yte[i])

import pandas as pd
log_df = pd.DataFrame(shadow_log)
agreement = (log_df.champ_pred == log_df.chal_pred).mean()
champ_acc = (log_df.champ_pred == log_df.truth).mean()
chal_acc = (log_df.chal_pred == log_df.truth).mean()
print(f'agreement champion-challenger: {agreement:.2%}')
print(f'accuracy champion:   {champ_acc:.4f}')
print(f'accuracy challenger: {chal_acc:.4f}  ({"🟢 better" if chal_acc > champ_acc else "🔴 worse"})')
print(f'latency champion:   p99={log_df.champ_lat_ms.quantile(0.99):.2f} ms')
print(f'latency challenger: p99={log_df.chal_lat_ms.quantile(0.99):.2f} ms')

## 2. Canary release con sticky assignment

In [ ]:
def sticky_bucket(user_id: str, percent: int) -> str:
    """Hash determinista → 'challenger' si está en el primer `percent`% del espacio."""
    h = int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100
    return 'challenger' if h < percent else 'champion'

# Simular 1000 users con canary al 10%
metrics = defaultdict(lambda: {'n': 0, 'correct': 0, 'lat_sum_ms': 0})
for i in range(1000):
    user_id = f'user_{i}'
    bucket = sticky_bucket(user_id, percent=10)
    model = challenger if bucket == 'challenger' else champion
    row = Xte[i % len(Xte)]
    t0 = time.perf_counter()
    pred = int(model.predict(row.reshape(1, -1))[0])
    lat = (time.perf_counter() - t0) * 1000
    truth = int(yte[i % len(yte)])
    metrics[bucket]['n'] += 1
    metrics[bucket]['correct'] += int(pred == truth)
    metrics[bucket]['lat_sum_ms'] += lat

for b in ['champion', 'challenger']:
    m = metrics[b]
    print(f'{b}: n={m["n"]}, acc={m["correct"] / m["n"]:.4f}, lat_mean={m["lat_sum_ms"] / m["n"]:.2f} ms')

## 3. Auto-rollback simulado

In [ ]:
def health_check_and_maybe_rollback(metrics_champion, metrics_challenger, lat_p99_tolerance=1.2, err_max=0.05):
    """Devuelve nuevo canary % (0 = rollback)."""
    c_lat = metrics_champion['lat_p99_ms']
    h_lat = metrics_challenger['lat_p99_ms']
    h_err = metrics_challenger['error_rate']
    if h_lat > c_lat * lat_p99_tolerance:
        return 0, f'ROLLBACK: lat_p99 challenger {h_lat:.1f} > {lat_p99_tolerance}× champion {c_lat:.1f}'
    if h_err > err_max:
        return 0, f'ROLLBACK: error rate challenger {h_err:.2%} > {err_max:.2%}'
    return None, 'OK'

# Caso 1: challenger sano
print(health_check_and_maybe_rollback({'lat_p99_ms': 8, 'error_rate': 0.01}, {'lat_p99_ms': 9, 'error_rate': 0.012}))
# Caso 2: challenger lento
print(health_check_and_maybe_rollback({'lat_p99_ms': 8, 'error_rate': 0.01}, {'lat_p99_ms': 25, 'error_rate': 0.01}))
# Caso 3: challenger con errores
print(health_check_and_maybe_rollback({'lat_p99_ms': 8, 'error_rate': 0.01}, {'lat_p99_ms': 9, 'error_rate': 0.08}))

## 4. A/B test riguroso — sample size y p-value

In [ ]:
from scipy import stats

def sample_size_for_proportions(p1, p2, alpha=0.05, power=0.8):
    """Tamaño por arm para detectar diff p2-p1 con dado poder."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    p_bar = (p1 + p2) / 2
    n = ((z_alpha * (2 * p_bar * (1 - p_bar)) ** 0.5 + z_beta * (p1 * (1 - p1) + p2 * (1 - p2)) ** 0.5) ** 2) / (p2 - p1) ** 2
    return int(np.ceil(n))

n_per_arm = sample_size_for_proportions(p1=0.92, p2=0.94)
print(f'sample size para detectar 92% → 94% con α=0.05 power=0.8: {n_per_arm} por arm')

# Simulamos los resultados con N exacto
rng = np.random.default_rng(0)
champ_outcomes = rng.binomial(1, 0.92, n_per_arm)
chal_outcomes = rng.binomial(1, 0.94, n_per_arm)
from statsmodels.stats.proportion import proportions_ztest
z, p = proportions_ztest([chal_outcomes.sum(), champ_outcomes.sum()], [n_per_arm, n_per_arm])
print(f'observado: champ {champ_outcomes.mean():.4f}, chal {chal_outcomes.mean():.4f}')
print(f'z={z:.3f}, p={p:.4f}  →  {"SIGNIFICATIVO" if p < 0.05 else "NO significativo"}')

## Ejercicio guiado

1. Convertí el shadow log a un análisis de **disagreement triage**: para las filas donde champion y challenger difieren, ¿cuál acierta más?
2. Implementá canary progresivo: empezá 1% → 5% → 25% → 100% con health checks entre cada escalón.
3. Agregá un guardrail: si `business_kpi_proxy` (ej. `proba > 0.7` rate) cae > 10%, no escales el canary.
4. Calculá sample size para tu caso real: detectar 1 punto de diff con `power=0.8`. ¿Cuántos días de tráfico necesitás?
5. Bonus: implementá esto en Istio con `VirtualService` y verificá distribución real de tráfico.

## Conclusiones

- Shadow = sin riesgo, costo 2×. Canary = riesgo limitado, costo 1×.
- Sticky assignment es no-negociable; sin él, A/B test es ruido.
- Rollback automático debe ser parte del deploy, no "plan B manual".
- Sample size pre-calculado vs ad-hoc es la diferencia entre decisión basada en evidencia y theater.